# Indice denso (bge-m3 + FAISS) en Kaggle

Version alternativa de `indice_denso_colab.ipynb` para **Kaggle Notebooks**. Flujo: **A (local)** empaqueta -> **B (Kaggle, GPU)** codifica -> **C (local)** integra.
La parte A y la C son las mismas celdas del notebook de Colab (`indice_denso_colab.ipynb`): el zip que genera la celda A de ese notebook es el que se sube aqui.

Una sola configuracion congelada: el `cache_emb` final debe venir de CUDA (la cache de v1 ya lo es). No mezclar con shards de MPS.

### Preparacion en Kaggle (una vez)
1. Cuenta verificada con telefono (si no, Kaggle no deja activar Internet ni GPU).
2. **Subir el zip como Dataset:** Create -> New Dataset -> arrastrar `data_corpus/colab_paquete.zip` (~155 MB) -> nombre `syntax-paquete` -> privado -> Create. Kaggle descomprime el zip solo; la celda 2 funciona con zip o carpeta.
3. **Nuevo Notebook** (Create -> New Notebook) e importar este `.ipynb` (File -> Import Notebook).
4. Panel derecho **Session options**: Accelerator = **GPU T4 x2** (evitar P100: sus ruedas de torch recientes no la soportan), **Internet = On** (bge-m3 se baja de Hugging Face), Persistence = Files only.
5. **Add Input** -> tu dataset `syntax-paquete` (queda en `/kaggle/input/syntax-paquete/`).

Cuotas: ~30 h/semana de GPU y 12 h por sesion.

In [ ]:
# KAGGLE. Verifica GPU e Internet. Usa una sola GPU (cuda:0): el codificador no reparte entre dos.
import os, urllib.request, torch
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
assert torch.cuda.is_available(), "Activar GPU en Session options (T4 x2)"
print(torch.cuda.get_device_name(0))
urllib.request.urlopen("https://huggingface.co", timeout=10)   # falla si Internet esta apagado
print("internet ok")

In [ ]:
# KAGGLE. Copia el paquete a /kaggle/working/syntax (zip o carpeta ya descomprimida) y verifica el corpus congelado.
import hashlib, shutil, zipfile
from pathlib import Path
ENTRADA = Path("/kaggle/input")
DEST = Path("/kaggle/working/syntax")
zips = list(ENTRADA.rglob("colab_paquete.zip"))
cfgs = list(ENTRADA.rglob("src/config.py"))
if DEST.exists():
    shutil.rmtree(DEST)
if zips:
    zipfile.ZipFile(zips[0]).extractall(DEST)
elif cfgs:
    shutil.copytree(cfgs[0].parent.parent, DEST)
else:
    raise SystemExit("No se encontro el paquete en /kaggle/input: agregar el dataset (Add Input)")
sha = hashlib.sha256((DEST / "data_corpus/indice/chunks.jsonl").read_bytes()).hexdigest()
assert sha == "27274f9b5f85c26177c22969dffb13288cf69e78ad0df2c48f35648942369d38", sha
print("chunks.jsonl ok;", len(list((DEST / "data_corpus/cache_emb").rglob("*.npz"))), "shards en cache")

In [ ]:
# KAGGLE. Dependencias (torch ya viene instalado; no reinstalarlo).
!pip -q install sentence-transformers==6.1.0 bm25s==0.3.11 faiss-cpu==1.15.1 numpy

In [ ]:
# KAGGLE. Codificar solo los fragmentos sin vector en cache_emb (de 47.966). Cada 512 se guarda un shard;
# si la sesion se corta, volver a correr (los shards estan en /kaggle/working si hubo Persistence/commit; si no, se pierden).
# Si hay OOM, bajar --batch a 8.
%cd /kaggle/working/syntax
!CUDA_VISIBLE_DEVICES=0 PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True SYNTAX_DEVICE=cuda PYTHONUNBUFFERED=1 python src/indexacion/construir_indice.py --batch 16 --no-manifest

In [ ]:
# KAGGLE. Empaquetar el resultado. Luego descargarlo: panel derecho Data -> Output -> /kaggle/working/resultado_indice.zip -> menu ... -> Download.
!cd /kaggle/working/syntax/data_corpus && zip -qr /kaggle/working/resultado_indice.zip cache_emb indice/index.faiss indice/indice_info.json
!ls -lh /kaggle/working/resultado_indice.zip

## Descargar y seguir en local

- **Sesion interactiva:** la celda anterior deja el zip en `/kaggle/working`; bajalo desde el panel Data -> Output.
- **Si te quedas sin sesion o prefieres no esperar:** Save Version -> *Save & Run All (Commit)* con Internet On y GPU. Corre todo en segundo plano y el zip queda en la pestaña **Output** de la version, aunque cierres la ventana.
- Luego, en local, mueve `resultado_indice.zip` a `data_corpus/` y corre la **celda C (10)** de `indice_denso_colab.ipynb`: reemplaza `cache_emb` y reconstruye `index.faiss`, BM25 y el manifest desde cache en segundos.
- Medir: `python src/evaluacion/retrieval_eval.py --modo bm25 denso hibrido --experimento e03_corpus_v2` y `python src/validaciones/manifest.py`.

### Repartir entre cuentas
Si se agota la cuota de una cuenta, otra persona repite los pasos con `--particion I/N` en la celda de codificacion (I = 1..N, mismo N), descarga su `cache_emb` y se juntan todos con la celda E3 de `indice_denso_colab.ipynb`.